# Prepare RSA Inputs and Outputs for DeepAlgPro

This notebook exports the DeepAlgPro train and test sequences as separate FASTA files for NetSurfP-3.0 and parses NetSurfP RSA predictions into a JSON mapping for later model training.

Workflow:
1. Export `data/deepalgpro_train_cleaned.csv` and `data/deepalgpro_test_cleaned.csv` to NetSurfP-ready FASTA files.
2. Split the train submission automatically into chunks that respect the NetSurfP server sequence limit.
3. Run NetSurfP-3.0 externally on each exported FASTA file.
3. Parse the NetSurfP output into `data/rsa/deepalgpro_rsa.json`.

In [ ]:
from pathlib import Path
import math
import pandas as pd
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


repo_root = find_repo_root(Path.cwd().resolve())
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from xallergen.rsa_preprocessing import export_deepalgpro_for_rsa, parse_netsurfp_rsa

train_csv = repo_root / "data/deepalgpro_train_cleaned.csv"
test_csv = repo_root / "data/deepalgpro_test_cleaned.csv"
rsa_dir = repo_root / "data/rsa"
rsa_json_path = repo_root / "data/rsa/deepalgpro_rsa.json"
train_sequence_count = len(pd.read_csv(train_csv))
netsurfp_train_chunk_size = math.ceil(train_sequence_count / 2)

train_csv, test_csv, rsa_dir, rsa_json_path, train_sequence_count, netsurfp_train_chunk_size

## Step 1: Export FASTA for NetSurfP

This cell validates the cleaned train/test CSVs, checks for duplicate `sequence_id` values across splits, reports exact duplicate sequences if any are present, splits the training set into two halves, and writes the FASTA files you should submit.

In [ ]:
export_summary = export_deepalgpro_for_rsa(
    train_csv=train_csv,
    test_csv=test_csv,
    output_dir=rsa_dir,
    train_chunk_size=netsurfp_train_chunk_size,
)
export_summary

Run NetSurfP-3.0 externally on all exported FASTA files before continuing:
- `data/rsa/deepalgpro_train_part1_for_netsurfp.fasta`
- `data/rsa/deepalgpro_train_part2_for_netsurfp.fasta`
- `data/rsa/deepalgpro_test_for_netsurfp.fasta`

With the current train size, this yields a near-even split across the two train FASTA files.

Set `netsurfp_input` below to either:
- a directory containing all train/test NetSurfP output tables, or
- a parent directory containing all NetSurfP output tables you want parsed together

The parser expects residue-level RSA values and validates full coverage against the original DeepAlgPro train/test splits.

Accepted RSA column names are inferred automatically from common NetSurfP-style names such as `rsa` and `rel_sasa`.

In [ ]:
# Update this to the directory containing the actual NetSurfP output files
# for all exported train/test FASTA submissions.
# Examples:
# netsurfp_input = repo_root / "data/rsa/netsurfp_outputs"
# netsurfp_input = repo_root / "data/rsa"
netsurfp_input = repo_root / "data/rsa/netsurfp_output"
netsurfp_input

## Step 2: Parse NetSurfP RSA Output

This cell validates that every sequence from the train/test splits has exactly one RSA vector, that each vector length matches the underlying sequence length, and that all RSA values lie in `[0, 1]`.

In [ ]:
if not netsurfp_input.exists():
    print(
        "Set `netsurfp_input` to your real NetSurfP output file or directory before running this cell.\n"
        f"Current placeholder path does not exist: {netsurfp_input}"
    )
else:
    parse_summary = parse_netsurfp_rsa(
        netsurfp_input=netsurfp_input,
        train_csv=train_csv,
        test_csv=test_csv,
        output_json=rsa_json_path,
    )
    parse_summary